# inplace-param-update — worked example 2: Out-of-place rebind never reaches the model

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `inplace-param-update`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When you write `param = param - lr * grad` you only rebind a local Python name; the `nn.Linear` that owns the parameter still references the original tensor, unchanged. The fix is to mutate `param.data` in place so the very object the module holds is modified.

## Worked solution

We run both the wrong and right updates on the same fresh layer to expose the difference.

1. **Snapshot the truth.** Record `model.weight`'s value and `data_ptr()` before touching anything.
2. **Wrong update.** `wrong_update` does `param = param - lr * grad` and returns the rebound local. The returned tensor holds the new values, but `model.weight` is unchanged — proof that the model never saw the step.
3. **Right update.** `right_update` does `param.data -= lr * grad`. Now `model.weight` itself moves, and its `data_ptr()` is identical to before because we mutated existing storage.
4. **Compare.** We print that the wrong path left the model weight equal to its original snapshot, while the right path changed it with a stable pointer.

This isolates the single line that decides whether a hand-rolled optimizer works at all.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(1)

def wrong_update(param, grad, lr):
    param = param - lr * grad   # rebinds local name only
    return param

def right_update(param, grad, lr):
    param.data -= lr * grad     # mutates the model's own storage

model = nn.Linear(4, 4, bias=False)
grad = t.ones_like(model.weight)
orig = model.weight.detach().clone()
ptr0 = model.weight.data_ptr()

rebound = wrong_update(model.weight, grad, 0.5)
print('wrong: model unchanged:', t.allclose(model.weight, orig),
      '| returned-local changed:', not t.allclose(rebound, orig))

right_update(model.weight, grad, 0.5)
print('right: model changed:', not t.allclose(model.weight, orig),
      '| ptr stable:', model.weight.data_ptr() == ptr0)